# Test WFS Data Bundler

**Goal**: Test the new `WFSDataBundler` class that abstracts WFS data collection across multiple regions.

**What this does**:
1. Fetches WFS data for multiple scope regions using `DataCollector` under the hood
2. Aggregates and deduplicates data across regions
3. Saves to GeoPackage with proper naming: `wfs_{service}_{layer}`
4. Provides checkpointing support for long runs

**Why use this**:
- Cleaner code - no notebook boilerplate
- Reusable - can be called from scripts or other notebooks
- Production-ready - includes error handling, logging, retry logic
- Scalable - designed for 12,130 regions

## Setup

In [19]:
import sys
sys.path.append("../../")

import src.paths as PATHS
import src.data.wfs_bundler as WFS_BUNDLER
import src.data.config as DATA_CONFIG

import geopandas as gpd
from pathlib import Path
from datetime import datetime
import shutil
import logging

# Setup logging to see bundler output
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

print("✅ Setup complete!")

✅ Setup complete!


## 1. Load Scope Regions

We'll start with a small test to verify everything works.

In [20]:
# Choose your test dataset
# Option 1: Full phase 2 dataset (12,130 regions)
filename = "wocu_output_fase2_v4"

# Option 2: Phase 1 dataset (smaller - 233 regions)
# filename = "phase1_2025-08-14_v1"

# Option 3: Luke's test set (11 regions - fastest)
# filename = "luke_inputs_v3"

gpkg_path = PATHS.DATA_DIR / f"{filename}.gpkg"

# Create output file for WFS data (clear naming: _w_wfs = with WFS layers)
output_path = PATHS.DATA_DIR / f"{filename}_w_wfs.gpkg"

# Copy original file to preserve base layers (vlakken_scope, middenlijn, etc.)
if not output_path.exists():
    shutil.copy2(gpkg_path, output_path)
    print(f"📁 Created working copy: {output_path.name}")
else:
    print(f"📁 Using existing file: {output_path.name}")

# Load scope regions
scope_regions = gpd.read_file(output_path, layer="vlakken_scope")

print(f"\n📍 Loaded {len(scope_regions)} scope regions")
print(f"📐 CRS: {scope_regions.crs}")

# Find the ID column (different geopackages use different names)
id_column = None
for col in ['position_id', 'location_id', 'region_id', 'id']:
    if col in scope_regions.columns:
        id_column = col
        break

if id_column:
    print(f"\n🗺️ ID column: '{id_column}'")
    print(f"   First 3 IDs: {scope_regions[id_column].head(3).tolist()}")
else:
    print(f"\n🗺️ No standard ID column found - will use generated IDs (region_0, region_1, ...)")
    print(f"   Available columns: {list(scope_regions.columns)}")

scope_regions.head()

📁 Created working copy: wocu_output_fase2_v4_w_wfs.gpkg

📍 Loaded 12130 scope regions
📐 CRS: EPSG:28992

🗺️ ID column: 'position_id'
   First 3 IDs: ['rijn_l_10343_10350', 'rijn_r_10343_10350', 'rijn_l_10350_10360']


,position_id,waterlichaam,geometry
0,rijn_l_10343_10350,"Boven-Rijn, Waal, Boven-Merwede, Beneden-Merwe...","POLYGON ((119484.028 425575.146, 119552 425573..."
1,rijn_r_10343_10350,"Boven-Rijn, Waal, Boven-Merwede, Beneden-Merwe...","POLYGON ((119552 425573, 119484.028 425575.146..."
2,rijn_l_10350_10360,"Boven-Rijn, Waal, Boven-Merwede, Beneden-Merwe...","POLYGON ((119384.07 425578.303, 119484.028 425..."
3,rijn_r_10350_10360,"Boven-Rijn, Waal, Boven-Merwede, Beneden-Merwe...","POLYGON ((119484.028 425575.146, 119384.07 425..."
4,rijn_l_10360_10370,"Boven-Rijn, Waal, Boven-Merwede, Beneden-Merwe...","POLYGON ((119284.111 425581.46, 119384.07 4255..."


## 2. Configure WFS Services

Using the default configuration which includes:
- Land Use (BRP Gewaspercelen)
- Buildings (BAG)
- Vegetation Legger (3 layers: bomen, heggen, vegetatieklassen)

In [21]:
# Load the default configuration
config = DATA_CONFIG.DataConfiguration()

print("📡 WFS Services configured:")
for i, wfs_service in enumerate(config.known_wfs_services, 1):
    print(f"\n{i}. {wfs_service.name}")
    print(f"   URL: {wfs_service.url}")
    print(f"   Layers: {', '.join(wfs_service.relevant_layers)}")

print(f"\n🔧 Buffer distance: {config.prediction_region_buffer}m")

📡 WFS Services configured:

1. land_use
   URL: https://service.pdok.nl/rvo/brpgewaspercelen/wfs/v1_0
   Layers: BrpGewas

2. building_location
   URL: https://service.pdok.nl/lv/bag/wfs/v2_0
   Layers: bag:pand

3. vegetation
   URL: https://geo.rijkswaterstaat.nl/services/ogc/gdr/rws_vegetatielegger/ows?version=2.0.0
   Layers: rws_vegetatielegger:bomen, rws_vegetatielegger:heggen, rws_vegetatielegger:vegetatieklassen

🔧 Buffer distance: 10m


## 3. Initialize WFS Data Bundler

In [ ]:
# Create bundler instance
bundler = WFS_BUNDLER.WFSDataBundler(
    scope_regions=scope_regions,
    config=config,
    wfs_timeout=30,  # 30 seconds per WFS request
    max_retries=3,   # Retry failed requests 3 times
)

print("✅ WFSDataBundler initialized!")

INFO: Initialized WFSDataBundler with 12130 regions


✅ WFSDataBundler initialized!


## 4. Fetch WFS Data

**TEST MODE**: Start by fetching just 10 regions to verify everything works.

Expected timing:
- 10 regions × 5 layers ≈ 30-60 seconds
- 100 regions × 5 layers ≈ 5-10 minutes
- 233 regions × 5 layers ≈ 10-20 minutes
- 12,130 regions × 5 layers ≈ 7-10 hours

In [24]:
# Fetch data - start with TEST MODE!
wfs_data, region_ids, successful, failed = bundler.fetch_all_regions(
    test_mode=False,        # Change to False for full run
    show_progress=True,    # Show progress bar

    
)

print("\n" + "="*50)
print("✅ Fetch complete!")
print("="*50)

INFO: Processing 12130 scope regions...
INFO: DataCollector: 30s timeout, 3 retries per service
Fetching WFS data:   0%|          | 0/12130 [00:00<?, ?it/s]INFO: Getting data from the WFS service land_use.
INFO: Getting data from the layer BrpGewas in land_use
INFO: Getting features 0 to 1.
INFO: Getting data from the WFS service building_location.
INFO: Getting data from the layer bag:pand in building_location
INFO: Getting data from the WFS service vegetation.
INFO: Getting data from the layer rws_vegetatielegger:bomen in vegetation
INFO: Getting data from the layer rws_vegetatielegger:heggen in vegetation
INFO: Getting data from the layer rws_vegetatielegger:vegetatieklassen in vegetation
INFO: Getting features 0 to 6.
Fetching WFS data:   0%|          | 1/12130 [00:01<5:45:44,  1.71s/it]INFO: Getting data from the WFS service land_use.
INFO: Getting data from the layer BrpGewas in land_use
INFO: Getting data from the WFS service building_location.
INFO: Getting data from the layer 


✅ Fetch complete!


## 5. View Summary Statistics

In [25]:
# Get detailed statistics
stats = bundler.get_summary_stats()

print("📊 Summary Statistics:")
print(f"\n  Regions processed: {stats['num_regions_processed']}")
print(f"  Successful: {stats['num_successful']}")
print(f"  Failed: {stats['num_failed']}")
print(f"  Services: {stats['num_services']}")

print("\n📡 Data by Service:")
for service_name, service_stats in stats['services'].items():
    print(f"\n  {service_name}:")
    print(f"    Layers: {service_stats['num_layers']}")
    for layer_name, layer_stats in service_stats['layers'].items():
        print(f"      - {layer_name}: {layer_stats['total_features']} features from {layer_stats['num_regions']} regions")

📊 Summary Statistics:

  Regions processed: 12130
  Successful: 12130
  Failed: 0
  Services: 3

📡 Data by Service:

  land_use:
    Layers: 1
      - BrpGewas: 36478 features from 9509 regions

  building_location:
    Layers: 1
      - bag:pand: 41047 features from 2643 regions

  vegetation:
    Layers: 3
      - rws_vegetatielegger:bomen: 34242 features from 7021 regions
      - rws_vegetatielegger:heggen: 5250 features from 1399 regions
      - rws_vegetatielegger:vegetatieklassen: 117816 features from 12104 regions


## 6. Save to GeoPackage

This will:
- Deduplicate geometries that span multiple regions
- Add `scope_region_id` column to track which region fetched each feature
- Save with naming convention: `wfs_{service}_{layer}`

In [26]:
# Save to GeoPackage
saved_layers = bundler.save_to_geopackage(
    output_path=output_path,
    add_region_ids=True,  # Track which region each feature came from
)

print("\n" + "="*50)
print(f"✅ Saved {len(saved_layers)} layers!")
print("="*50)

INFO: Saving WFS data to wocu_output_fase2_v4_w_wfs.gpkg...
INFO: Processing service: land_use
INFO: Created 10,397 records
INFO: Saved 10397 features to layer: land_use/BrpGewas (removed 26081 duplicates, 71.5%)
INFO: Processing service: building_location
INFO: Created 23,564 records
INFO: Saved 23564 features to layer: building_location/bag:pand (removed 17483 duplicates, 42.6%)
INFO: Processing service: vegetation
INFO: Created 17,802 records
INFO: Saved 17802 features to layer: vegetation/rws_vegetatielegger:bomen (removed 16440 duplicates, 48.0%)
INFO: Created 2,700 records
INFO: Saved 2700 features to layer: vegetation/rws_vegetatielegger:heggen (removed 2550 duplicates, 48.6%)
INFO: Created 28,869 records
INFO: Saved 28869 features to layer: vegetation/rws_vegetatielegger:vegetatieklassen (removed 88947 duplicates, 75.5%)
INFO: DONE! Saved 5 new layers to wocu_output_fase2_v4_w_wfs.gpkg
INFO: New layers:
INFO:   - land_use/BrpGewas
INFO:   - building_location/bag:pand
INFO:   - 


✅ Saved 5 layers!


## 7. Verify Saved Layers

In [27]:
# List all layers in the GeoPackage
layers = bundler.list_geopackage_layers(output_path)

print(f"📦 Layers in {output_path.name}:\n")

print(f"📂 Original layers ({len(layers['original'])})")
for layer in layers['original']:
    print(f"  - {layer}")

print(f"\n🆕 WFS layers ({len(layers['wfs'])})")
for layer in layers['wfs']:
    print(f"  - {layer}")

📦 Layers in wocu_output_fase2_v4_w_wfs.gpkg:

📂 Original layers (9)
  - building_location/bag:pand
  - centrelines
  - land_use/BrpGewas
  - punten_oever
  - vegetation/rws_vegetatielegger:bomen
  - vegetation/rws_vegetatielegger:heggen
  - vegetation/rws_vegetatielegger:vegetatieklassen
  - vlakken_erosie
  - vlakken_scope

🆕 WFS layers (0)


## 8. Inspect a Sample Layer

In [28]:
# Pick the first WFS layer to inspect
if saved_layers:
    sample_layer = saved_layers[0]
    print(f"🔍 Inspecting layer: {sample_layer}\n")
    
    sample_gdf = gpd.read_file(output_path, layer=sample_layer)
    
    print(f"📊 Shape: {sample_gdf.shape}")
    print(f"📐 CRS: {sample_gdf.crs}")
    print(f"\n🗂️ Columns: {list(sample_gdf.columns)}")
    print(f"\n📍 First 5 rows:")
    display(sample_gdf.head())
else:
    print("⚠️ No layers were saved")

🔍 Inspecting layer: land_use/BrpGewas

📊 Shape: (10397, 7)
📐 CRS: EPSG:28992

🗂️ Columns: ['category', 'gewas', 'gewascode', 'jaar', 'status', 'scope_region_id', 'geometry']

📍 First 5 rows:


,category,gewas,gewascode,jaar,status,scope_region_id,geometry
0,Grasland,"Grasland, natuurlijk. Met landbouwactiviteiten.",331,2024,Definitief,rijn_l_10343_10350,"POLYGON ((119309.156 425105.137, 119323.692 42..."
1,Landschapselement,Sloot,343,2024,Definitief,rijn_l_10350_10360,"POLYGON ((119308.095 425403.812, 119308.967 42..."
2,Grasland,"Grasland, natuurlijk. Met landbouwactiviteiten.",331,2024,Definitief,rijn_r_10350_10360,"POLYGON ((119163.27 424642.566, 119165.109 424..."
3,Grasland,"Grasland, blijvend",265,2024,Definitief,rijn_l_10360_10370,"POLYGON ((119125.351 425297.779, 119124.478 42..."
4,Grasland,"Grasland, natuurlijk. Met landbouwactiviteiten.",331,2024,Definitief,rijn_l_10360_10370,"POLYGON ((119066.491 424965.809, 119065.414 42..."


---

## ✅ Next Steps

### If test run succeeded:

1. **Open in QGIS**: Load the GeoPackage and visually inspect the `wfs_*` layers
2. **Verify data quality**: 
   - Do geometries look correct?
   - Are they in the right locations?
   - Compare with scope regions layer
3. **Scale up gradually**:
   - Try 100 regions (set `num_test=100`)
   - Try 500 regions to estimate full run time
   - Run full dataset overnight (set `test_mode=False`)

### For production runs (12,130 regions):

1. **Run overnight** - estimated 7-10 hours
2. **Monitor progress** - check logs for failures
3. **Validate in QGIS** before using in DataHandler
4. **Then implement** `DataHandler.load_remote_data_from_geopackage()`

### Expected output layers:
- `wfs_land_use_BrpGewas` - Agricultural crop parcels
- `wfs_building_location_bag_pand` - Building footprints
- `wfs_vegetation_rws_vegetatielegger_bomen` - Trees
- `wfs_vegetation_rws_vegetatielegger_heggen` - Hedges
- `wfs_vegetation_rws_vegetatielegger_vegetatieklassen` - Vegetation classes